## Section 1 — Setup + data loading

Load `data/merged.csv` and cluster CSVs (produced by `dawn_data_creation.ipynb`). Paths are set in `dawn_config.py`.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from dawn_config import DATA_DIR

merged = pd.read_csv(DATA_DIR / "merged.csv")
clusters = {
    0: pd.read_csv(DATA_DIR / "cluster0_MI_MN_WI.csv"),
    1: pd.read_csv(DATA_DIR / "cluster1_IN_KS_MO_NE_OH.csv"),
    2: pd.read_csv(DATA_DIR / "cluster2_IA_IL.csv"),
}

for name, df in [("merged", merged)] + [(f"cluster_{k}", v) for k, v in clusters.items()]:
    years = sorted(df["year"].unique()) if "year" in df.columns else []
    n_counties = df["county_name"].nunique() if "county_name" in df.columns else 0
    print(f"{name}: shape={df.shape}, years={years[0]}..{years[-1]}, #counties={n_counties}")

## Section 2 — Tuning (optional)

Before or alongside training, you can optimize:

1. **Season date ranges** — which (start, end) windows for spring, summer, fall maximize R².
2. **Hyperparameters** — `clear_thresh` (precip threshold for “clear day”) and `anomaly_percentile` (percentiles for prcp, tmax, tmin, snow).

Implementation lives in `dawn_tuning.py`. Run per cluster; results can be copied into `CLUSTER_CONFIGS` in `dawn_models.py`.

### 2a. Binary search for season dates

Optimize spring (start/end), then summer (holding spring fixed), then fall (holding spring+summer fixed). Progressively finer steps.

In [ ]:
from dawn_tuning import run_binary_search_all_seasons

# Example: tune date ranges for one cluster (slow — many RF fits)
cluster_id = 0
df_cluster = clusters[cluster_id]
tuned_config, best_r2 = run_binary_search_all_seasons(
    df_cluster,
    clear_thresh=2.0,
    anomaly_percentile={"prcp": 77, "tmax": 95, "tmin": 91, "snow": 84},
    verbose=True,
)
print(f"Cluster {cluster_id} best R²: {best_r2:.4f}")
print("Tuned config (spring, summer, fall):", tuned_config["spring"], tuned_config["summer"], tuned_config["fall"])

### 2b. Random search over season dates

Sample random (spring, summer, fall) date ranges; keep the config with highest R².

In [ ]:
from dawn_tuning import random_search_optimize_dates

# Example: random search for one cluster
cluster_id = 0
df_cluster = clusters[cluster_id]
best_config, best_r2, results = random_search_optimize_dates(
    df_cluster,
    num_iterations=50,
    clear_thresh=2.0,
    anomaly_percentile={"prcp": 77, "tmax": 95, "tmin": 91, "snow": 84},
    verbose=True,
)
print(f"Cluster {cluster_id} best R²: {best_r2:.4f}")

### 2c. Hyperparameter tuning: clear_thresh + anomaly_percentile

Tune the clear-day precip threshold and the percentile used for anomaly counts (prcp, tmax, tmin, snow). Use a fixed date config (e.g. from 2a/2b or defaults).

In [ ]:
from dawn_tuning import tune_hyperparameters

# Example: tune clear_thresh and anomaly percentiles for one cluster
cluster_id = 0
df_cluster = clusters[cluster_id]
# Optionally pass date_config from binary/random search; here use defaults
best_config, best_r2 = tune_hyperparameters(
    df_cluster,
    date_config=None,
    clear_thresh_values=(1.0, 2.0, 2.5, 3.0, 3.5, 5.0),
    anomaly_percentile_grid=None,
    num_random_samples=40,
    use_random=True,
    verbose=True,
)
print(f"Cluster {cluster_id} best R²: {best_r2:.4f}")
print("Best clear_thresh:", best_config["clear_thresh"])
print("Best anomaly_percentile:", best_config["anomaly_percentile"])

## Section 3 — Feature engineering

Features are built per cluster from daily weather + yield:

- **Seasonal means**: tmax/tmin averaged over spring, summer, fall windows
- **Precip/snow sums**: total prcp and snow per season
- **Anomaly counts**: days per season above percentile thresholds (prcp, tmax, tmin, snow)
- **Clear-day counts**: days per season with precip ≤ threshold
- **Optional**: PCA components on the above (MICE-imputed) for dimensionality reduction

Implementation lives in `dawn_features.py`. Here we only call:

In [ ]:
from dawn_features import build_features
from dawn_models import CLUSTER_CONFIGS

# Example: build features for one cluster
cluster_id = 0
df_cluster = clusters[cluster_id]
config = CLUSTER_CONFIGS[cluster_id]
X, y = build_features(df_cluster, config)
print(f"Cluster {cluster_id}: X.shape={X.shape}, y.shape={y.shape}")

## Section 4 — Models (baseline → final)

Model candidates tested:
1. **Baseline RF** — raw tmax/tmin/prcp/snow aggregates (no seasonal engineering)
2. **RF + engineered features** — same RF on seasonal + anomaly + clear-day features; chosen as best
3. **Final model** — residual stacking (RF + MLP on residuals)

All use cluster-specific tuned configs (season windows, clear_thresh, anomaly percentiles).

In [ ]:
from dawn_models import run_baseline_rf

for cid, df in clusters.items():
    r2, mse = run_baseline_rf(df)
    print(f"Cluster {cid} (baseline RF): R²={r2:.3f}, MSE={mse:.2f}")

In [ ]:
from dawn_models import run_rf_engineered, CLUSTER_CONFIGS

for cid, df in clusters.items():
    config = CLUSTER_CONFIGS[cid]
    r2, mse = run_rf_engineered(df, config)
    print(f"Cluster {cid} (RF + engineered): R²={r2:.3f}, MSE={mse:.2f}")

In [ ]:
from dawn_models import run_final_model, CLUSTER_CONFIGS

for cid, df in clusters.items():
    config = CLUSTER_CONFIGS[cid]
    r2, mse = run_final_model(df, config)
    print(f"Cluster {cid} (final — residual stacking): R²={r2:.3f}, MSE={mse:.2f}")

## Section 5 — Final evaluation

| Cluster | Model | R² | MSE |
|---------|-------|-----|------|
| 0 | RF + Engineered | 0.764 | 53.62 |
| 1 | RF + Engineered | 0.692 | 122.24 |
| 2 | RF + Engineered | 0.772 | 86.46 |

*(Desktop dawn_models run - Random Forest with cluster-tuned features.)*